# Week 10 Lab: Graphs - BFS, DFS, and Topological Sort

## Objectives

By the end of this lab, you should be able to:

1. Understand and implement Breadth-First Search (BFS) and Depth-First Search (DFS) on graphs.
2. Visualize traversals on real-world style graphs using the `networkx` library.
3. Compare recursive and iterative versions of DFS.
4. Apply graph traversal to solve **practical, complex problems**.
5. Implement and use **Topological Sort** in practical scheduling scenarios.

In [ ]:
!pip install networkx matplotlib

# Part A: Warm-up with Graph Construction and Visualization

### Task 1: Constructing and Visualizing Graphs
1. Create an undirected graph with at least 10 nodes and 15 edges using `networkx`.
2. Assign weights to edges (random integers between 1 and 10).
3. Visualize the graph with edge labels.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random

def create_and_visualize_graph():
    G = nx.Graph()
    G.add_nodes_from(range(1, 11))
    
    edges = set()
    while len(edges) < 15:
        u = random.randint(1, 10)
        v = random.randint(1, 10)
        if u != v:
            edges.add((u, v))
    for u, v in edges:
        G.add_edge(u, v, weight=random.randint(1, 10))
    
    pos = nx.spring_layout(G)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=800, font_size=10)
    labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels)
    plt.show()
    return G

# Part B: Implementing BFS and DFS

### Task 2: Implement BFS

- Implement BFS traversal from a given source node.
- Print the traversal order.

In [ ]:
from collections import deque

def bfs(graph, start_node):
    visited = set()
    queue = deque([start_node])
    traversal = []
    
    while queue:
        node = queue.popleft()
        if node not in visited:
            visited.add(node)
            traversal.append(node)
            for neighbor in graph.neighbors(node):
                if neighbor not in visited:
                    queue.append(neighbor)
    return traversal

### Task 3: Implement Recursive DFS

- Implement Recursive DFS.
- Print the traversal order.

In [ ]:
def dfs_recursive(graph, start_node, visited=None):
    if visited is None:
        visited = set()
    visited.add(start_node)
    traversal = [start_node]
    for neighbor in graph.neighbors(start_node):
        if neighbor not in visited:
            traversal.extend(dfs_recursive(graph, neighbor, visited))
    return traversal

### Task 4: Implement Iterative DFS

- Implement Iterative DFS using a stack.
- Compare traversal order with recursive DFS.

In [ ]:
def dfs_iterative(graph, start_node):
    visited = set()
    stack = [start_node]
    traversal = []
    
    while stack:
        node = stack.pop()
        if node not in visited:
            visited.add(node)
            traversal.append(node)
            stack.extend(sorted(graph.neighbors(node), reverse=True))
    return traversal

# Part C: Visualizing BFS and DFS

### Task 5: Stepwise Visualization

- Write a function to visualize the step-by-step traversal of BFS and DFS on a given graph.
- At each step, highlight the current node in a different color.

In [ ]:
import time

def visualize_bfs_dfs(graph, start_node, method="bfs"):
    if method == "bfs":
        traversal = bfs(graph, start_node)
    else:
        traversal = dfs_iterative(graph, start_node)
    
    pos = nx.spring_layout(graph)
    for idx, node in enumerate(traversal):
        plt.figure()
        nx.draw(graph, pos, with_labels=True, node_color='lightgray', node_size=800)
        nx.draw_networkx_nodes(graph, pos, nodelist=[node], node_color='red')
        plt.title(f"Step {idx+1}: Visiting {node}")
        plt.show()
        time.sleep(0.5)

# Part D: Applications of BFS/DFS

### Task 6: Connected Components

- Use DFS to find all connected components in an undirected graph.
- Visualize each component in a different color.

In [ ]:
def find_connected_components(graph):
    visited = set()
    components = []
    
    for node in graph.nodes():
        if node not in visited:
            component = dfs_recursive(graph, node, visited)
            components.append(component)
    return components

### Task 7: Cycle Detection

- Use DFS to detect if a graph contains a cycle.
- Apply on both directed and undirected graphs.

In [ ]:
def detect_cycle(graph, directed=False):
    visited = set()
    parent = {}
    
    def dfs_undirected(node, par):
        visited.add(node)
        for neighbor in graph.neighbors(node):
            if neighbor not in visited:
                parent[neighbor] = node
                if dfs_undirected(neighbor, node):
                    return True
            elif parent[node] != neighbor:
                return True
        return False
    
    def dfs_directed(node, rec_stack):
        visited.add(node)
        rec_stack.add(node)
        for neighbor in graph.neighbors(node):
            if neighbor not in visited:
                if dfs_directed(neighbor, rec_stack):
                    return True
            elif neighbor in rec_stack:
                return True
        rec_stack.remove(node)
        return False
    
    if directed:
        for node in graph.nodes():
            if node not in visited:
                if dfs_directed(node, set()):
                    return True
        return False
    else:
        for node in graph.nodes():
            if node not in visited:
                if dfs_undirected(node, -1):
                    return True
        return False

### Task 8: Shortest Path in Unweighted Graph

- Use BFS to compute the shortest path between two nodes.
- Visualize the shortest path highlighted in the graph.

In [ ]:
def shortest_path_unweighted(graph, source, target):
    visited = set()
    queue = deque([(source, [source])])
    
    while queue:
        node, path = queue.popleft()
        if node == target:
            return path
        visited.add(node)
        for neighbor in graph.neighbors(node):
            if neighbor not in visited:
                queue.append((neighbor, path + [neighbor]))
    return None

# Part E: Topological Sorting

### Task 9: Implement Topological Sort

- Implement Kahn’s Algorithm (BFS-based) for topological sorting.
- Test on a Directed Acyclic Graph (DAG).

In [ ]:
def topological_sort_kahn(graph):
    in_degree = {node: 0 for node in graph.nodes()}
    for u in graph.nodes():
        for v in graph.neighbors(u):
            in_degree[v] += 1
    
    queue = deque([node for node in graph.nodes() if in_degree[node] == 0])
    topo_order = []
    
    while queue:
        node = queue.popleft()
        topo_order.append(node)
        for neighbor in graph.neighbors(node):
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)
    
    if len(topo_order) == len(graph.nodes()):
        return topo_order
    else:
        return None 

### Task 10: Course Scheduling Problem (Real-world Application)

- Suppose there are 10 courses with prerequisite constraints.
- Model the courses as a graph and use topological sort to determine a valid order to take the courses.
- If no valid ordering exists, print that the schedule is impossible.

In [ ]:
def course_schedule(courses, prerequisites):
    G = nx.DiGraph()
    G.add_nodes_from(courses)
    G.add_edges_from(prerequisites)
    
    order = topological_sort_kahn(G)
    if order is None:
        print("No valid course schedule exists (cycle detected).")
    else:
        print("A valid course schedule is:", order)
    return order